# Day 07: Modular Production Pipeline Architecture & Reusable ETL Functions
> **Part 7 of 10 in the PySpark Production Data Engineering Series**

[![PySpark](https://img.shields.io/badge/PySpark-3.5%2B%20%2F%204.0%2B-F26522?style=for-the-badge&logo=apachespark&logoColor=white)](https://spark.apache.org/)
[![Python](https://img.shields.io/badge/Python-3.10%2B-3776AB?style=for-the-badge&logo=python&logoColor=white)](https://python.org)
[![Status](https://img.shields.io/badge/Module-Day%2007%20of%2010-008080?style=for-the-badge)](../README.md)

---

## 🏗️ Day 07: Modular Production Pipeline Architecture & Reusable ETL Functions
### 🎯 Learning Objectives:
1. Refactor monolithic script code into modular, testable, and reusable Python functions.
2. Build dedicated single-responsibility components:
   - `ingest_csv()`: Standardized ingestion with schema enforcement.
   - `add_metadata()`: Lineage and ingestion timestamps.
   - `validate_orders()`: Contract validation returning valid and rejected tuples.
   - `deduplicate_orders()`: Idempotent deduplication.
   - `transform_orders()`: Business feature engineering.
   - `create_product_report()`: Gold analytical mart generation.
   - `write_parquet()`: Reusable partitioned persistence layer.
3. Create a master pipeline driver `run_pipeline(spark, source_path)` that orchestrates the workflow.
4. Output structured data quality metrics and verify contract compliance.
---


### ⚙️ Initialize SparkSession for Day 07


In [234]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

import pandas as pd
from pathlib import Path
import logging
import os

# Ensure HADOOP_HOME is set for Windows
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] += os.pathsep + r'C:\hadoop\bin'

spark = (
    SparkSession.builder
    .appName("Day7_Reusable_ETL")
    .master("local[*]")
    .getOrCreate()
)

BASE = Path("day7_project")

for folder in ["source", "bronze", "silver", "gold", "rejected"]:
    (BASE / folder).mkdir(parents=True, exist_ok=True)

print("Environment ready")


Environment ready


In [235]:
data = {
    "order_id": [
        1001, 1002, 1003, 1004, 1005,
        1006, 1007, 1008, 1008, 1009
    ],
    "customer_id": [
        101, 102, 103, 104, 105,
        101, None, 106, 106, 107
    ],
    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor",
        "Laptop", "Mouse", "Keyboard",
        "Monitor", "Monitor", "Laptop"
    ],
    "quantity": [1, 2, 1, 2, -1, 3, 1, 2, 2, 1],
    "price": [
        55000, 800, 1500, 12000, 55000,
        800, 1500, 12000, 12000, 55000
    ],
    "order_date": [
        "2026-10-01", "2026-10-01", "2026-10-02",
        "2026-10-02", "2026-10-03", "2026-10-03",
        "2026-10-03", "2026-10-04", "2026-10-04",
        "INVALID"
    ]
}

df_orders = pd.DataFrame(data)
df_orders["customer_id"] = df_orders["customer_id"].astype("Int64")

df_orders.to_csv(
    BASE / "source" / "orders.csv",
    index=False
)

print("Practice CSV created")


Practice CSV created


In [236]:
ORDER_SCHEMA = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("order_date", StringType(), True)
])

In [237]:
def ingest_csv(spark, file_path, schema):
    df = (
        spark.read
        .option("header", True)
        .schema(schema)
        .csv(str(file_path))
    )

    return df

In [238]:
raw_df = ingest_csv(
    spark,
    BASE / "source" / "orders.csv",
    ORDER_SCHEMA
)

raw_df.show()

+--------+-----------+--------+--------+-------+----------+
|order_id|customer_id| product|quantity|  price|order_date|
+--------+-----------+--------+--------+-------+----------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|
|    1005|        105|  Laptop|      -1|55000.0|2026-10-03|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|
|    1007|       NULL|Keyboard|       1| 1500.0|2026-10-03|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|
|    1009|        107|  Laptop|       1|55000.0|   INVALID|
+--------+-----------+--------+--------+-------+----------+



In [239]:
def add_metadata(df):
    return (
        df
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_file",
            F.input_file_name()
        )
    )

In [240]:
bronze_df = add_metadata(raw_df)

bronze_df.show(truncate=False)

+--------+-----------+--------+--------+-------+----------+--------------------------+---------------------------------------------------------------------------------------+
|order_id|customer_id|product |quantity|price  |order_date|ingestion_timestamp       |source_file                                                                            |
+--------+-----------+--------+--------+-------+----------+--------------------------+---------------------------------------------------------------------------------------+
|1001    |101        |Laptop  |1       |55000.0|2026-10-01|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%20Learning/Pyspark/day7_project/source/orders.csv|
|1002    |102        |Mouse   |2       |800.0  |2026-10-01|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%20Learning/Pyspark/day7_project/source/orders.csv|
|1003    |103        |Keyboard|1       |1500.0 |2026-10-02|2026-10-08 07:43:24.430005|file:///C:/Users/Me/Documents/Machine%2

In [241]:
def validate_orders(df):

    checked = (
        df
        .withColumn(
            "parsed_date",
            F.try_to_timestamp(
                F.col("order_date"),
                F.lit("yyyy-MM-dd")
            ).cast("date")
        )
        .withColumn(
            "validation_errors",
            F.array_compact(
                F.array(
                    F.when(
                        F.col("order_id").isNull(),
                        F.lit("Missing order ID")
                    ),
                    F.when(
                        F.col("customer_id").isNull(),
                        F.lit("Missing customer ID")
                    ),
                    F.when(
                        F.col("quantity").isNull() |
                        (F.col("quantity") <= 0),
                        F.lit("Invalid quantity")
                    ),
                    F.when(
                        F.col("price").isNull() |
                        (F.col("price") <= 0),
                        F.lit("Invalid price")
                    ),
                    F.when(
                        F.col("parsed_date").isNull(),
                        F.lit("Invalid date")
                    )
                )
            )
        )
    )

    valid = checked.filter(
        F.size("validation_errors") == 0
    )

    rejected = checked.filter(
        F.size("validation_errors") > 0
    )

    return valid, rejected

In [242]:
valid_df, rejected_df = validate_orders(bronze_df)

valid_df.show()
rejected_df.select(
    "order_id",
    "validation_errors"
).show(truncate=False)

+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-----------+-----------------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|         source_file|parsed_date|validation_errors|
+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-----------+-----------------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-01|               []|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-01|               []|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-02|               []|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 2026-10-02|               []|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|2026-10-08 07:43:

In [243]:
def deduplicate_orders(df):

    window = (
        Window
        .partitionBy("order_id")
        .orderBy(
            F.col("ingestion_timestamp").desc(),
            F.col("source_file").asc()
        )
    )

    ranked = df.withColumn(
        "rn",
        F.row_number().over(window)
    )

    unique = ranked.filter(F.col("rn") == 1).drop("rn")

    duplicates = ranked.filter(F.col("rn") > 1).drop("rn")

    return unique, duplicates

In [244]:
unique_df, duplicates_df = deduplicate_orders(valid_df)

print("Unique:", unique_df.count())
print("Duplicates:", duplicates_df.count())

Unique: 6
Duplicates: 1


In [245]:
def transform_orders(df):
    return (
        df
        .withColumn(
            "product",
            F.initcap(F.trim(F.col("product")))
        )
        .withColumn(
            "order_date",
            F.col("parsed_date")
        )
        .withColumn(
            "revenue",
            F.col("quantity") * F.col("price")
        )
        .drop("parsed_date", "validation_errors")
    )

In [246]:
silver_df = transform_orders(unique_df)

silver_df.select(
    "order_id",
    "product",
    "quantity",
    "price",
    "revenue"
).show()

+--------+--------+--------+-------+-------+
|order_id| product|quantity|  price|revenue|
+--------+--------+--------+-------+-------+
|    1001|  Laptop|       1|55000.0|55000.0|
|    1002|   Mouse|       2|  800.0| 1600.0|
|    1003|Keyboard|       1| 1500.0| 1500.0|
|    1004| Monitor|       2|12000.0|24000.0|
|    1006|   Mouse|       3|  800.0| 2400.0|
|    1008| Monitor|       2|12000.0|24000.0|
+--------+--------+--------+-------+-------+



In [247]:
def write_parquet(df, path, partitions=None):

    writer = df.write.mode("overwrite")

    if partitions:
        writer = writer.partitionBy(*partitions)

    writer.parquet(str(path))

In [248]:
def create_product_report(df):
    return (
        df
        .groupBy("product")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum("quantity").alias("total_quantity"),
            F.sum("revenue").alias("total_revenue"),
            F.avg("revenue").alias("avg_order_value")
        )
        .orderBy(F.desc("total_revenue"))
    )

In [249]:
gold_df = create_product_report(silver_df)

gold_df.show()

+--------+------------+--------------+-------------+---------------+
| product|total_orders|total_quantity|total_revenue|avg_order_value|
+--------+------------+--------------+-------------+---------------+
|  Laptop|           1|             1|      55000.0|        55000.0|
| Monitor|           2|             4|      48000.0|        24000.0|
|   Mouse|           2|             5|       4000.0|         2000.0|
|Keyboard|           1|             1|       1500.0|         1500.0|
+--------+------------+--------------+-------------+---------------+



### 🚀 Master Pipeline Driver Function: `run_pipeline`


In [250]:
def run_pipeline(spark, source_path):

    # 1. Ingestion
    raw = ingest_csv(
        spark,
        source_path,
        ORDER_SCHEMA
    )

    # 2. Metadata
    bronze = add_metadata(raw)

    # 3. Validation
    valid, rejected = validate_orders(bronze)

    # 4. Deduplication
    unique, duplicates = deduplicate_orders(valid)

    # 5. Transformation
    silver = transform_orders(unique)

    # 6. Aggregation
    gold = create_product_report(silver)

    # 7. Calculate metrics
    total = bronze.count()
    invalid_count = rejected.count()
    duplicate_count = duplicates.count()
    accepted = silver.count()

    assert total == (
        invalid_count + duplicate_count + accepted
    ), "Record reconciliation failed"

    # 8. Save outputs
    write_parquet(bronze, BASE / "bronze")
    write_parquet(rejected, BASE / "rejected" / "invalid")
    write_parquet(duplicates, BASE / "rejected" / "duplicates")
    write_parquet(silver, BASE / "silver")
    write_parquet(gold, BASE / "gold")

    return {
        "total": total,
        "accepted": accepted,
        "invalid": invalid_count,
        "duplicates": duplicate_count
    }

In [251]:
report = run_pipeline(
    spark,
    BASE / "source" / "orders.csv"
)

print("DATA QUALITY REPORT")

for key, value in report.items():
    print(f"{key}: {value}")

DATA QUALITY REPORT
total: 10
accepted: 6
invalid: 3
duplicates: 1


In [252]:
saved_silver = spark.read.parquet(
    str(BASE / "silver")
)

saved_gold = spark.read.parquet(
    str(BASE / "gold")
)

saved_silver.show()
saved_gold.show()

+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-------+
|order_id|customer_id| product|quantity|  price|order_date| ingestion_timestamp|         source_file|revenue|
+--------+-----------+--------+--------+-------+----------+--------------------+--------------------+-------+
|    1001|        101|  Laptop|       1|55000.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...|55000.0|
|    1002|        102|   Mouse|       2|  800.0|2026-10-01|2026-10-08 07:43:...|file:///C:/Users/...| 1600.0|
|    1003|        103|Keyboard|       1| 1500.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...| 1500.0|
|    1004|        104| Monitor|       2|12000.0|2026-10-02|2026-10-08 07:43:...|file:///C:/Users/...|24000.0|
|    1006|        101|   Mouse|       3|  800.0|2026-10-03|2026-10-08 07:43:...|file:///C:/Users/...| 2400.0|
|    1008|        106| Monitor|       2|12000.0|2026-10-04|2026-10-08 07:43:...|file:///C:/Users/...|24000.0|
+--------+

---
### 🎓 Day 07 Complete!
Great job completing this module!
👉 **Next Up:** [**Day 08: Enterprise Pipeline Engineering, Logging & Automated Testing**](./Day_08_Enterprise_Logging_Config_and_Automated_Testing.ipynb)
